# Tulpar LoRA birleştirme (Gemma 4 26B-A4B) — Google Drive ile

Eğitim bitti; `tulpar-lora` (uzman katmanları dahil) llama.cpp'de `--lora` ile **takılamıyor**. Bu defter LoRA'yı taban modele **birleştirip** HP'nin kullandığı küçük GGUF'a çevirir.

**Çalışma zamanı:** A100 (yüksek RAM, ~83 GB). Merge CPU RAM'de yapılır, GPU şart değil ama yüksek RAM şart.
**Drive'da ~120 GB boş yer gerekir** (birleşmiş 16-bit model ~52 GB + f16 GGUF ~52 GB + Q3 çıktı ~13 GB). Ara dosyalar işin sonunda silinir.

**Önce yap:** HP'deki `~/tulpar/models/tulpar-lora/` klasörünü Drive'a `MyDrive/tulpar/tulpar-lora/` olarak yükle (2 GB).
Sırayla her hücreyi çalıştır.

In [ ]:
# 1) Drive'ı bağla: /content/drive altında Drive'ın dosya sistemi gibi görünür; yazdığımız dosyalar Colab kapansa da kalır
from google.colab import drive
drive.mount("/content/drive")
import os
IS = "/content/drive/MyDrive/tulpar"
os.makedirs(IS, exist_ok = True)
assert os.path.exists(f"{IS}/tulpar-lora/adapter_model.safetensors"), "tulpar-lora klasörünü Drive'da MyDrive/tulpar/ altına yükle"
!df -h /content /content/drive | cat
!free -g | head -2

In [ ]:
# 2) Kurulum (eğitimdeki aynı Unsloth; uzman katmanlı LoRA'yı doğru birleştiren o)
%%capture
!pip install -U unsloth

In [ ]:
# 3) LoRA'yı Unsloth ile yükle ve 16-bit olarak birleştirip Drive'a yaz (~15-25 dk)
import os, gc
from unsloth import FastModel
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
model, tok = FastModel.from_pretrained(f"{IS}/tulpar-lora", max_seq_length = 2048, load_in_4bit = True)
model.save_pretrained_merged(f"{IS}/birlesik", tok, save_method = "merged_16bit")
del model; gc.collect()
!ls -lh {IS}/birlesik | tail -5
!rm -rf /root/.cache/huggingface

In [ ]:
# 4) llama.cpp: GGUF'a çevir (bf16, Drive'a) ve nicemleyiciyi derle
import shutil
from huggingface_hub import hf_hub_download
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp > /dev/null 2>&1
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!pip install -q -U "transformers>=5" huggingface_hub   # üstteki gereksinim dosyası transformers'ı eskiye çekiyor, gemma4'ü tanımıyor
for f in ["tokenizer.json", "tokenizer_config.json", "chat_template.jinja"]:   # birleştirmenin yazdığı tokenizer ayarı yerine orijinali
    shutil.copy(hf_hub_download("unsloth/gemma-4-26B-A4B-it", f), f"{IS}/birlesik/{f}")
!python /content/llama.cpp/convert_hf_to_gguf.py {IS}/birlesik --outtype bf16 --outfile {IS}/tulpar-birlesik-bf16.gguf
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build /content/llama.cpp/build --target llama-quantize -j 8 2>&1 | tail -2


In [ ]:
# 5) HP'nin kullandığı boyuta nicele (Q3_K_M ~13 GB; HP RAM'i 16 GB). Sonuç Drive'da.
!/content/llama.cpp/build/bin/llama-quantize {IS}/tulpar-birlesik-bf16.gguf {IS}/tulpar-gemma-Q3_K_M.gguf Q3_K_M
!ls -lh {IS}/*.gguf

In [ ]:
# 6) Ara dosyaları sil (Drive'da yer aç). Sadece son GGUF kalır.
!rm -rf {IS}/birlesik {IS}/tulpar-birlesik-bf16.gguf
!ls -lh {IS}

## 7) HP'ye al
`tulpar-gemma-Q3_K_M.gguf` dosyasını Drive'dan indir (tarayıcıdan sağ tık → İndir) ya da Pi/HP'den `rclone`/tarayıcıyla çek, `~/tulpar/models/` altına koy.
Sonra Claude'a söyle: HP'de `hizli.env` modeli bu dosyaya çevirip llama-server'ı yeniden başlatır ve eski modelle karşılaştırırız.